# 00 What Is Fine Tuning?

## Overview

This notebook explains what fine tuning is and where it fits in the wider AI workflow.
You will compare pretraining, prompting, retrieval augmented generation, and fine tuning.
You will also run a small text generation demo with `distilgpt2`.

Estimated time: 20 to 30 minutes.

## Prerequisites

You only need basic Python and notebook familiarity.
No machine learning background is required.

In [ ]:
%pip install -q transformers==4.40.0 torch==2.2.2

## Pretraining, Fine Tuning, and Prompting

Pretraining is the expensive first stage where a model learns general language patterns from a massive dataset.

Fine tuning is the follow up stage where you keep training that pretrained model on a narrower dataset so it becomes better at a specific job.

Prompting does not change the model weights at all.
It changes the instructions you give the model at inference time.

A simple mental model works well here:

* A pretrained model is a generalist.
* A fine tuned model is a specialist.
* A well written prompt is a better briefing for the same worker.

## When Fine Tuning Is the Right Tool

Fine tuning is a good fit when you need the model to answer in a repeatable style, follow a strict format, or internalize task specific behavior across many examples.

If your problem is missing knowledge, retrieval augmented generation is often better.
If your problem is vague instructions, prompt engineering is usually the first thing to try.

## Visual Analogy

```text
pretrained model
generalist
    |
    | small, task specific dataset
    v
fine tuned model
specialist
```

The base model starts broad.
Fine tuning narrows it toward the exact pattern you want.

## Zero Code Style Demo

We are not training in this notebook.
Instead, we use one small pretrained model to show how a general model responds to the same topic before and after stronger task framing.

This is a teaching shortcut.
In a real fine tune, the model weights would change.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

def generate(prompt: str, max_new_tokens: int = 40) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

In [ ]:
before_prompt = "Explain what overfitting is in one short paragraph."
after_prompt = (
    "You are a beginner friendly machine learning tutor. "
    "Explain what overfitting is in one short paragraph using plain English and one simple analogy."
)

before_text = generate(before_prompt)
after_text = generate(after_prompt)

print("Prompt only:\n")
print(before_text)
print("\nStronger task framing:\n")
print(after_text)

## Key Takeaways

* Pretraining builds broad capability.
* Fine tuning narrows that capability toward one job.
* Prompting changes instructions, not model weights.
* Fine tuning is useful for repeatable behavior and strict output formats.
* RAG is often better when the issue is missing knowledge, not missing behavior.

## Next Steps

Continue to [01 Choosing a Base Model](../01-choosing-a-base-model/01-choosing-a-base-model.ipynb).